<a href="https://colab.research.google.com/github/Youssif-Kady/flayrank_task1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract Answers (1/2)
- **One Row Grain:** One row represents a single unique `(url, date)` record for our lane.
- **Time Window:** Mid-panel observation window (`2026-03-01` to `2026-03-31`).
- **Tables Used:** HuggingFace Parquet dataset (`FlyRank/internship-warehouse`).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Contract Answers (2/2) & Feature Frame
- **Label / Target:** `clicks` (or downstream ranking proxy).
- **Features (Max 5):**
  1. `clicks_7d_avg`: knowable at the decision moment because it only uses data strictly up to day T-1.
  2. `impressions_30d_sum`: knowable at the decision moment because it aggregates historical logs prior to prediction time.
  3. `past_ctr`: knowable at the decision moment because it is derived from historical clicks and impressions.
  4. `days_active`: knowable at the decision moment because metadata is logged upon record creation.
  5. `avg_position_7d`: knowable at the decision moment because position trends are recorded daily prior to prediction.
- **Deliberately Excluded:** Raw query strings and future-dated indicators to avoid leakage and unnecessary bloat.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import os
import polars as pl
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download, login

# 1. Force overwrite the cached token in environment
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

REPO_ID = "FlyRank/internship-warehouse"

# 2. Discover March 2026 dataset files dynamically
print("Discovering dataset repository files...")
api = HfApi(token=hf_token)
all_repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")

march_remote_files = [f for f in all_repo_files if "2026-03" in f and f.endswith(".parquet")]

if not march_remote_files:
    march_remote_files = [f for f in all_repo_files if f.endswith(".parquet")][:5]

print(f"Found {len(march_remote_files)} matching file(s): {march_remote_files}")

# 3. Download the verified file
local_downloaded_files = []
for remote_file in march_remote_files:
    path = hf_hub_download(
        repo_id=REPO_ID,
        filename=remote_file,
        repo_type="dataset",
        token=hf_token
    )
    local_downloaded_files.append(path)

# 4. Load into Polars DataFrame
df_march = pl.read_parquet(local_downloaded_files)
print("✅ Data successfully loaded!")

# Query 1: Verify Grain Uniqueness using actual schema columns
# Checking uniqueness on (content_hash_id, report_date)
duplicates = df_march.group_by(["content_hash_id", "report_date"]).len().filter(pl.col("len") > 1)

print(f"Total Duplicate Rows Found: {len(duplicates)}")
if len(duplicates) == 0:
    print("✅ Grain Proof Passed: One row really is a unique (content_hash_id, report_date) pair.")
else:
    print(duplicates)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Discovering dataset repository files...
Found 1 matching file(s): ['fact_content_daily_performance/month=2026-03/data_0.parquet']
✅ Data successfully loaded!
Total Duplicate Rows Found: 0
✅ Grain Proof Passed: One row really is a unique (content_hash_id, report_date) pair.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# Query 2: Row Count & Date Span for Mid-Panel Month (2026-03)
total_rows = len(df_march)
min_date = df_march["report_date"].min()
max_date = df_march["report_date"].max()

print("--- Query 2 Results ---")
print(f"Total Rows Count : {total_rows}")
print(f"Start Date       : {min_date}")
print(f"End Date         : {max_date}")

--- Query 2 Results ---
Total Rows Count : 9841378
Start Date       : 2026-03-01
End Date         : 2026-03-31


### Named Limitation
**Limitation:** Our lane's slice lacks real-time user session context prior to indexing. Cold-start URLs with zero historical search logs during `2026-03` will produce missing/zero values for rolling historical signals, making short-term predictions reliant on fallback defaults.

In [ ]:
# Query 3: Availability check for required key signals
if "gsc_clicks" in df_march.columns:
    available_rows = df_march.filter(pl.col("gsc_clicks").is_not_null()).height
else:
    available_rows = total_rows

availability_rate = (available_rows / total_rows) * 100

print("--- Query 3 Results ---")
print(f"Total Rows       : {total_rows}")
print(f"Surviving Rows   : {available_rows}")
print(f"Availability Rate: {availability_rate:.2f}%")

--- Query 3 Results ---
Total Rows       : 9841378
Surviving Rows   : 9841378
Availability Rate: 100.00%


In [ ]:
import pandas as pd
from sklearn.metrics import r2_score

# Convert a sample slice to pandas for feature engineering
df_sample = df_march.head(10000).to_pandas()

# Fill missing target values with 0
df_sample['gsc_clicks'] = df_sample['gsc_clicks'].fillna(0)
df_sample['gsc_impressions'] = df_sample['gsc_impressions'].fillna(0)

# Build Honest Feature
df_sample['feat_clicks_7d_avg'] = df_sample['gsc_clicks'].shift(1).rolling(7, min_periods=1).mean().fillna(0)

# STEP A: Inject Leakage Feature (The Trap)
df_sample['leaked_feature'] = df_sample['gsc_clicks'] * 0.999 + 0.001

score_leaked = r2_score(df_sample['gsc_clicks'], df_sample['leaked_feature'])
print(f"⚠️ Leaked Model Score (Near Perfect Trap): {score_leaked:.4f}")

# STEP B: Remove Leakage Feature & Measure Honest Baseline
df_sample.drop(columns=['leaked_feature'], inplace=True)

score_honest = r2_score(df_sample['gsc_clicks'], df_sample['feat_clicks_7d_avg'])
print(f"✅ Honest Baseline Score (No Leakage): {score_honest:.4f}")

⚠️ Leaked Model Score (Near Perfect Trap): 1.0000
✅ Honest Baseline Score (No Leakage): -0.1099


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.